In [1]:
!pip install torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 22.4 MB/s eta 0:00:00


0.84385

In [34]:
import pandas as pd
import numpy as np
import networkx as nx
import xgboost as xgb
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from math import log
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# 1. CHARGEMENT ET RÉINDEXATION DES DONNÉES
# ==========================================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

path = "/content/drive/MyDrive/MLNS/"
print("Chargement des données...")
node_info = pd.read_csv(path + 'node_information.csv', header=None, index_col=0)
train_data = pd.read_csv(path + 'train.txt', sep=' ', header=None, names=['source', 'target', 'label'])
test_data = pd.read_csv(path + 'test.txt', sep=' ', header=None, names=['source', 'target'])

print(f"Nœuds: {len(node_info)}, Train: {len(train_data)}, Test: {len(test_data)}")

print("Mappage des IDs de nœuds...")
mapping = {original_id: new_id for new_id, original_id in enumerate(node_info.index)}

train_data['source'] = train_data['source'].map(mapping)
train_data['target'] = train_data['target'].map(mapping)
test_data['source'] = test_data['source'].map(mapping)
test_data['target'] = test_data['target'].map(mapping)

train_data = train_data.dropna().astype(int)
test_data = test_data.dropna().astype(int)

# ==========================================
# 2. RÉDUCTION DE DIMENSION (SVD BOOSTÉE)
# ==========================================
N_COMPONENTS = 128
print(f"\nRéduction des features textuelles avec SVD ({N_COMPONENTS} composantes)...")
svd = TruncatedSVD(n_components=N_COMPONENTS, random_state=42)
node_features_reduced = svd.fit_transform(node_info)

print(f"Features réduites: {node_features_reduced.shape}")
print(f"Variance expliquée: {svd.explained_variance_ratio_.sum():.4f}")

# Dictionnaires pour un accès ultra-rapide
svd_dict = {i: node_features_reduced[i] for i in range(len(node_features_reduced))}
raw_dict = {i: node_info.values[i] for i in range(len(node_info))}

# ==========================================
# 3. EXTRACTION DES FEATURES (TEXTE BRUT + SVD + TOPO)
# ==========================================
print("\nPréparation de l'extraction de features...")

def extract_ultimate_features(df, graph, svd_features, raw_features):
    features = []
    degrees = dict(graph.degree())
    pagerank = nx.pagerank(graph, alpha=0.85, max_iter=100)

    for i, row in df.iterrows():
        u, v = int(row['source']), int(row['target'])

        # --- Features Topologiques ---
        common_neigh = 0
        jaccard = 0
        adamic_adar = 0
        ra_index = 0

        deg_u = degrees.get(u, 0)
        deg_v = degrees.get(v, 0)

        if graph.has_node(u) and graph.has_node(v):
            neighbors = list(nx.common_neighbors(graph, u, v))
            common_neigh = len(neighbors)
            union_size = len(set(graph[u]) | set(graph[v]))
            jaccard = common_neigh / union_size if union_size > 0 else 0

            for w in neighbors:
                d_w = degrees.get(w, 0)
                if d_w > 1:
                    adamic_adar += 1 / log(d_w)
                    ra_index += 1 / d_w

        pr_prod = pagerank.get(u, 0) * pagerank.get(v, 0)

        try:
            sp = nx.shortest_path_length(graph, u, v)
        except (nx.NetworkXNoPath, nx.NodeNotFound):
            sp = 10  # Valeur de pénalité si les nœuds sont déconnectés

        # --- Features SVD (Texte compressé) ---
        svd_u = svd_features[u]
        svd_v = svd_features[v]

        hadamard_svd = svd_u * svd_v
        diff_abs_svd = np.abs(svd_u - svd_v)
        svd_sum = svd_u + svd_v

        svd_cos = np.dot(svd_u, svd_v) / (np.linalg.norm(svd_u) * np.linalg.norm(svd_v) + 1e-8)
        svd_euclidean = np.linalg.norm(svd_u - svd_v)

        # --- Features Texte BRUT (La similarité exacte) ---
        raw_u = raw_features[u]
        raw_v = raw_features[v]

        raw_dot = np.dot(raw_u, raw_v)
        raw_cos = raw_dot / (np.linalg.norm(raw_u) * np.linalg.norm(raw_v) + 1e-8)

        # Assemblage
        base_feats = [common_neigh, jaccard, adamic_adar, ra_index, deg_u, deg_v, deg_u*deg_v, sp, pr_prod, svd_cos, svd_euclidean, raw_dot, raw_cos]

        features.append(np.concatenate([base_feats, hadamard_svd, diff_abs_svd, svd_sum]))

    return np.array(features)

# ==========================================
# 4. STRATIFIED K-FOLD & ENSEMBLING
# ==========================================
print("\n" + "="*50)
print("DÉBUT DU K-FOLD STRATIFIÉ")
print("="*50)

n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

y_train_full = train_data['label'].values
train_pairs = train_data[['source', 'target']].values

test_preds = np.zeros(len(test_data))
oof_preds = np.zeros(len(train_data))
num_total_nodes = len(node_features_reduced)

for fold, (train_idx, val_idx) in enumerate(skf.split(train_pairs, y_train_full)):
    print(f"\n{'='*50}")
    print(f"FOLD {fold + 1}/{n_splits}")
    print(f"{'='*50}")

    fold_train_edges = train_data.iloc[train_idx]
    fold_val_edges = train_data.iloc[val_idx]

    # Construction du graphe (Uniquement avec les arêtes positives du train !)
    G_fold = nx.Graph()
    G_fold.add_nodes_from(range(num_total_nodes))

    positive_train_edges = fold_train_edges[fold_train_edges['label'] == 1][['source', 'target']].values
    G_fold.add_edges_from(positive_train_edges)

    print(f"Graphe fold: {len(G_fold.nodes())} nœuds, {len(G_fold.edges())} arêtes")

    # ==========================================
    # EXTRAIRE FEATURES
    # ==========================================
    print("  Extraction features Train...")
    X_train_fold = extract_ultimate_features(fold_train_edges, G_fold, svd_dict, raw_dict)
    y_train_fold = fold_train_edges['label'].values

    print("  Extraction features Val...")
    X_val_fold = extract_ultimate_features(fold_val_edges, G_fold, svd_dict, raw_dict)
    y_val_fold = fold_val_edges['label'].values

    print(f"  Shapes: X_train {X_train_fold.shape}, X_val {X_val_fold.shape}")

    # ==========================================
    # ENTRAÎNER XGBOOST
    # ==========================================
    print("  Entraînement XGBoost...")

    clf = xgb.XGBClassifier(
        n_estimators=1500,
        max_depth=5,
        learning_rate=0.015,
        subsample=0.8,
        colsample_bytree=0.2, # Gardé très bas pour contrer le surapprentissage
        gamma=2,              # Filtre strict pour les nouvelles branches
        eval_metric='auc',
        early_stopping_rounds=100,
        random_state=42,
        n_jobs=-1
    )

    clf.fit(
        X_train_fold, y_train_fold,
        eval_set=[(X_val_fold, y_val_fold)],
        verbose=150
    )

    # Prédictions
    oof_preds[val_idx] = clf.predict_proba(X_val_fold)[:, 1]
    fold_auc = roc_auc_score(y_val_fold, oof_preds[val_idx])
    print(f"  Validation AUC: {fold_auc:.4f}")

    # ==========================================
    # PRÉDICTIONS TEST
    # ==========================================
    print("  Extraction features Test...")
    X_test_fold = extract_ultimate_features(test_data, G_fold, svd_dict, raw_dict)
    test_preds += clf.predict_proba(X_test_fold)[:, 1] / n_splits

    print(f"✓ FOLD {fold + 1} TERMINÉ")

# ==========================================
# 5. SOUMISSION
# ==========================================
print("\n" + "="*50)
print("GÉNÉRATION DE LA SOUMISSION")
print("="*50)

submission = pd.DataFrame({
    'ID': range(len(test_preds)),
    'Predicted': test_preds
})

submission.to_csv('submission_pure_xgboost_finale.csv', index=False)
print("\n✓ Fichier 'submission_pure_xgboost_finale.csv' généré !")

print("\n" + "="*50)
print("STATISTIQUES FINALES")
print("="*50)
final_auc = roc_auc_score(y_train_full, oof_preds)
print(f"OOF Global AUC: {final_auc:.4f}")
print(f"Test predictions - min: {test_preds.min():.4f}, max: {test_preds.max():.4f}, mean: {test_preds.mean():.4f}")

Mounted at /content/drive
Chargement des données...
Nœuds: 3597, Train: 10496, Test: 3498
Mappage des IDs de nœuds...

Réduction des features textuelles avec SVD (128 composantes)...
Features réduites: (3597, 128)
Variance expliquée: 0.7137

Préparation de l'extraction de features...

DÉBUT DU K-FOLD STRATIFIÉ

FOLD 1/5
Graphe fold: 3597 nœuds, 4198 arêtes
  Extraction features Train...
  Extraction features Val...
  Shapes: X_train (8396, 397), X_val (2100, 397)
  Entraînement XGBoost...
[0]	validation_0-auc:0.57837
[134]	validation_0-auc:0.64799
  Validation AUC: 0.6782
  Extraction features Test...
✓ FOLD 1 TERMINÉ

FOLD 2/5
Graphe fold: 3597 nœuds, 4198 arêtes
  Extraction features Train...
  Extraction features Val...
  Shapes: X_train (8397, 397), X_val (2099, 397)
  Entraînement XGBoost...
[0]	validation_0-auc:0.57300
[131]	validation_0-auc:0.64526
  Validation AUC: 0.6704
  Extraction features Test...
✓ FOLD 2 TERMINÉ

FOLD 3/5
Graphe fold: 3597 nœuds, 4198 arêtes
  Extraction 

0.84157

In [33]:
import pandas as pd
import numpy as np
import networkx as nx
import xgboost as xgb
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from math import log
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# 1. CHARGEMENT ET RÉINDEXATION DES DONNÉES
# ==========================================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

path = "/content/drive/MyDrive/MLNS/"
print("Chargement des données...")
node_info = pd.read_csv(path + 'node_information.csv', header=None, index_col=0)
train_data = pd.read_csv(path + 'train.txt', sep=' ', header=None, names=['source', 'target', 'label'])
test_data = pd.read_csv(path + 'test.txt', sep=' ', header=None, names=['source', 'target'])

print(f"Nœuds: {len(node_info)}, Train: {len(train_data)}, Test: {len(test_data)}")

print("Mappage des IDs de nœuds...")
mapping = {original_id: new_id for new_id, original_id in enumerate(node_info.index)}

train_data['source'] = train_data['source'].map(mapping)
train_data['target'] = train_data['target'].map(mapping)
test_data['source'] = test_data['source'].map(mapping)
test_data['target'] = test_data['target'].map(mapping)

train_data = train_data.dropna().astype(int)
test_data = test_data.dropna().astype(int)

# ==========================================
# 2. RÉDUCTION DE DIMENSION (SVD BOOSTÉE)
# ==========================================
N_COMPONENTS = 128
print(f"\nRéduction des features textuelles avec SVD ({N_COMPONENTS} composantes)...")
svd = TruncatedSVD(n_components=N_COMPONENTS, random_state=42)
node_features_reduced = svd.fit_transform(node_info)

print(f"Features réduites: {node_features_reduced.shape}")
print(f"Variance expliquée: {svd.explained_variance_ratio_.sum():.4f}")

svd_dict = {i: node_features_reduced[i] for i in range(len(node_features_reduced))}

# ==========================================
# 3. EXTRACTION DES FEATURES (100% MATH & TOPO)
# ==========================================
print("\nPréparation de l'extraction de features...")

def extract_ultimate_features(df, graph, svd_features):
    features = []
    degrees = dict(graph.degree())
    pagerank = nx.pagerank(graph, alpha=0.85, max_iter=100)

    for i, row in df.iterrows():
        u, v = int(row['source']), int(row['target'])

        # --- Features Topologiques ---
        common_neigh = 0
        jaccard = 0
        adamic_adar = 0
        ra_index = 0

        deg_u = degrees.get(u, 0)
        deg_v = degrees.get(v, 0)

        if graph.has_node(u) and graph.has_node(v):
            neighbors = list(nx.common_neighbors(graph, u, v))
            common_neigh = len(neighbors)
            union_size = len(set(graph[u]) | set(graph[v]))
            jaccard = common_neigh / union_size if union_size > 0 else 0

            for w in neighbors:
                d_w = degrees.get(w, 0)
                if d_w > 1:
                    adamic_adar += 1 / log(d_w)
                    ra_index += 1 / d_w

        pr_prod = pagerank.get(u, 0) * pagerank.get(v, 0)

        try:
            sp = nx.shortest_path_length(graph, u, v)
        except (nx.NetworkXNoPath, nx.NodeNotFound):
            sp = 10  # Valeur de pénalité si les nœuds sont déconnectés

        # --- Features SVD (Texte) ---
        svd_u = svd_features[u]
        svd_v = svd_features[v]

        # Interactions directes
        hadamard_svd = svd_u * svd_v
        diff_abs_svd = np.abs(svd_u - svd_v)

        # Distances globales sur les textes
        svd_cos = np.dot(svd_u, svd_v) / (np.linalg.norm(svd_u) * np.linalg.norm(svd_v) + 1e-8)
        svd_euclidean = np.linalg.norm(svd_u - svd_v)

        base_feats = [common_neigh, jaccard, adamic_adar, ra_index, deg_u, deg_v, deg_u*deg_v, sp, pr_prod, svd_cos, svd_euclidean]

        features.append(np.concatenate([base_feats, hadamard_svd, diff_abs_svd]))

    return np.array(features)

# ==========================================
# 4. STRATIFIED K-FOLD & ENSEMBLING
# ==========================================
print("\n" + "="*50)
print("DÉBUT DU K-FOLD STRATIFIÉ")
print("="*50)

n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

y_train_full = train_data['label'].values
train_pairs = train_data[['source', 'target']].values

test_preds = np.zeros(len(test_data))
oof_preds = np.zeros(len(train_data))
num_total_nodes = len(node_features_reduced)

for fold, (train_idx, val_idx) in enumerate(skf.split(train_pairs, y_train_full)):
    print(f"\n{'='*50}")
    print(f"FOLD {fold + 1}/{n_splits}")
    print(f"{'='*50}")

    fold_train_edges = train_data.iloc[train_idx]
    fold_val_edges = train_data.iloc[val_idx]

    # Construction du graphe (Uniquement avec les arêtes positives du train !)
    G_fold = nx.Graph()
    G_fold.add_nodes_from(range(num_total_nodes))

    positive_train_edges = fold_train_edges[fold_train_edges['label'] == 1][['source', 'target']].values
    G_fold.add_edges_from(positive_train_edges)

    print(f"Graphe fold: {len(G_fold.nodes())} nœuds, {len(G_fold.edges())} arêtes")

    # ==========================================
    # EXTRAIRE FEATURES
    # ==========================================
    print("  Extraction features Train...")
    X_train_fold = extract_ultimate_features(fold_train_edges, G_fold, svd_dict)
    y_train_fold = fold_train_edges['label'].values

    print("  Extraction features Val...")
    X_val_fold = extract_ultimate_features(fold_val_edges, G_fold, svd_dict)
    y_val_fold = fold_val_edges['label'].values

    print(f"  Shapes: X_train {X_train_fold.shape}, X_val {X_val_fold.shape}")

    # ==========================================
    # ENTRAÎNER XGBOOST
    # ==========================================
    print("  Entraînement XGBoost...")

    clf = xgb.XGBClassifier(
        n_estimators=1500,       # Plus d'arbres
        max_depth=5,             # Profondeur maîtrisée pour éviter le surapprentissage
        learning_rate=0.015,     # Apprentissage plus lent et robuste
        subsample=0.8,
        colsample_bytree=0.2,
        eval_metric='auc',
        early_stopping_rounds=100, # Patience augmentée
        random_state=42,
        n_jobs=-1,
        gamma = 2
    )

    clf.fit(
        X_train_fold, y_train_fold,
        eval_set=[(X_val_fold, y_val_fold)],
        verbose=150
    )

    # Prédictions
    oof_preds[val_idx] = clf.predict_proba(X_val_fold)[:, 1]
    fold_auc = roc_auc_score(y_val_fold, oof_preds[val_idx])
    print(f"  Validation AUC: {fold_auc:.4f}")

    # ==========================================
    # PRÉDICTIONS TEST
    # ==========================================
    print("  Extraction features Test...")
    X_test_fold = extract_ultimate_features(test_data, G_fold, svd_dict)
    test_preds += clf.predict_proba(X_test_fold)[:, 1] / n_splits

    print(f"✓ FOLD {fold + 1} TERMINÉ")

# ==========================================
# 5. SOUMISSION
# ==========================================
print("\n" + "="*50)
print("GÉNÉRATION DE LA SOUMISSION")
print("="*50)

submission = pd.DataFrame({
    'ID': range(len(test_preds)),
    'Predicted': test_preds
})

submission.to_csv('submission_pure_xgboost.csv', index=False)
print("\n✓ Fichier 'submission_pure_xgboost.csv' généré !")

print("\n" + "="*50)
print("STATISTIQUES FINALES")
print("="*50)
final_auc = roc_auc_score(y_train_full, oof_preds)
print(f"OOF Global AUC: {final_auc:.4f}")
print(f"Test predictions - min: {test_preds.min():.4f}, max: {test_preds.max():.4f}, mean: {test_preds.mean():.4f}")

Mounted at /content/drive
Chargement des données...
Nœuds: 3597, Train: 10496, Test: 3498
Mappage des IDs de nœuds...

Réduction des features textuelles avec SVD (128 composantes)...
Features réduites: (3597, 128)
Variance expliquée: 0.7137

Préparation de l'extraction de features...

DÉBUT DU K-FOLD STRATIFIÉ

FOLD 1/5
Graphe fold: 3597 nœuds, 4198 arêtes
  Extraction features Train...
  Extraction features Val...
  Shapes: X_train (8396, 267), X_val (2100, 267)
  Entraînement XGBoost...
[0]	validation_0-auc:0.55537
[138]	validation_0-auc:0.63213
  Validation AUC: 0.6443
  Extraction features Test...
✓ FOLD 1 TERMINÉ

FOLD 2/5
Graphe fold: 3597 nœuds, 4198 arêtes
  Extraction features Train...
  Extraction features Val...
  Shapes: X_train (8397, 267), X_val (2099, 267)
  Entraînement XGBoost...
[0]	validation_0-auc:0.60430
[138]	validation_0-auc:0.63166
  Validation AUC: 0.6443
  Extraction features Test...
✓ FOLD 2 TERMINÉ

FOLD 3/5
Graphe fold: 3597 nœuds, 4198 arêtes
  Extraction 

0.82549

In [21]:
import pandas as pd
import numpy as np
import networkx as nx
import xgboost as xgb
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import LGConv
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import StratifiedKFold
from math import log
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# 0. DEVICE SETUP
# ==========================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# ==========================================
# 1. CHARGEMENT ET RÉINDEXATION DES DONNÉES
# ==========================================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

path = "/content/drive/MyDrive/MLNS/"
print("Chargement des données...")
node_info = pd.read_csv(path + 'node_information.csv', header=None, index_col=0)
train_data = pd.read_csv(path + 'train.txt', sep=' ', header=None, names=['source', 'target', 'label'])
test_data = pd.read_csv(path + 'test.txt', sep=' ', header=None, names=['source', 'target'])

print(f"Nœuds: {len(node_info)}, Train: {len(train_data)}, Test: {len(test_data)}")

print("Mappage des IDs de nœuds pour PyTorch Geometric...")
mapping = {original_id: new_id for new_id, original_id in enumerate(node_info.index)}

train_data['source'] = train_data['source'].map(mapping)
train_data['target'] = train_data['target'].map(mapping)
test_data['source'] = test_data['source'].map(mapping)
test_data['target'] = test_data['target'].map(mapping)

train_data = train_data.dropna().astype(int)
test_data = test_data.dropna().astype(int)

# ==========================================
# 2. RÉDUCTION DE DIMENSION (SVD)
# ==========================================
print("Réduction des 932 features textuelles avec SVD...")
svd = TruncatedSVD(n_components=64, random_state=42)
node_features_reduced = svd.fit_transform(node_info)
node_features_tensor = torch.FloatTensor(node_features_reduced).to(device)

print(f"Features réduites: {node_features_reduced.shape}")
print(f"Variance expliquée: {svd.explained_variance_ratio_.sum():.4f}")

# ==========================================
# 3. GNN : LIGHTGCN
# ==========================================
print("\n" + "="*50)
print("INITIALISATION DU MODÈLE LIGHTGCN")
print("="*50)

class LightGCN_FeatureModel(nn.Module):
    """
    LightGCN adapté pour utiliser les features textuelles (SVD).

    Architecture:
    - 1 couche de projection linéaire (pour avoir des paramètres apprenables)
    - N couches de LGConv (propagation pure sans activation ni poids)
    - Moyenne des couches (E^0 + E^1 + ... + E^K)
    """
    def __init__(self, in_feats, out_feats, num_layers=3):
        super(LightGCN_FeatureModel, self).__init__()

        # Projection initiale de nos features SVD
        self.projection = nn.Linear(in_feats, out_feats)

        # Couches LightGCN (Propagation)
        self.convs = nn.ModuleList([LGConv() for _ in range(num_layers)])

    def forward(self, x, edge_index):
        # Etape 0: Projection des features
        x = self.projection(x)

        # Stockage des embeddings de chaque couche (E^0 à E^K)
        xs = [x]

        # Propagation via les couches LightGCN
        for conv in self.convs:
            x = conv(x, edge_index)
            xs.append(x)

        # Combinaison (moyenne) des embeddings de toutes les échelles
        out = torch.stack(xs, dim=0).mean(dim=0)

        return out


class LinkPredictor_Advanced(nn.Module):
    """
    Prédicteur de liens basé sur embeddings
    """
    def __init__(self, in_feats):
        super().__init__()
        self.W1 = nn.Linear(in_feats * 2, in_feats)
        self.W2 = nn.Linear(in_feats, 1)
        self.dropout = nn.Dropout(0.2)

    def forward(self, h_u, h_v):
        combined = torch.cat([h_u, h_v], dim=1)
        x = F.relu(self.W1(combined))
        x = self.dropout(x)
        return torch.sigmoid(self.W2(x))


# ==========================================
# 4. FONCTION D'ENTRAÎNEMENT LIGHTGCN
# ==========================================
def train_lightgcn_for_fold(
    node_features_tensor,
    edge_index,
    train_src_all,
    train_dst_all,
    labels_tensor,
    epochs=2000,
    lr=0.1,           # LR souvent un peu plus élevé avec LightGCN
    weight_decay=1e-3, # Decay plus fort conseillé avec LightGCN
    patience=100
):

    embed_model = LightGCN_FeatureModel(
        in_feats=64,
        out_feats=64,
        num_layers=3   # 3 ou 4 couches est l'optimum classique
    ).to(device)

    predictor = LinkPredictor_Advanced(64).to(device)

    optimizer = torch.optim.Adam(
        list(embed_model.parameters()) + list(predictor.parameters()),
        lr=lr,
        weight_decay=weight_decay,
        betas=(0.9, 0.999)
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=0.5,
        patience=20
    )

    embed_model.train()
    predictor.train()

    best_loss = float('inf')
    patience_counter = 0

    print("Entraînement LightGCN...")
    for epoch in range(epochs):
        h = embed_model(node_features_tensor, edge_index)
        pos_score = predictor(h[train_src_all], h[train_dst_all]).squeeze()
        loss = F.binary_cross_entropy(pos_score, labels_tensor)

        optimizer.zero_grad()
        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            list(embed_model.parameters()) + list(predictor.parameters()),
            max_norm=1.0
        )

        optimizer.step()
        scheduler.step(loss)

        if loss.item() < best_loss:
            best_loss = loss.item()
            patience_counter = 0
        else:
            patience_counter += 1

        if epoch % 200 == 0:
            print(f"  Epoch {epoch:04d} | Loss: {loss.item():.4f} | LR: {optimizer.param_groups[0]['lr']:.6f}")

        if patience_counter >= patience:
            print(f"  Early stopping à epoch {epoch}")
            break

    embed_model.eval()
    predictor.eval()

    with torch.no_grad():
        lightgcn_embeddings = embed_model(node_features_tensor, edge_index).cpu().numpy()

    return lightgcn_embeddings, embed_model, predictor


# ==========================================
# 5. EXTRACTION DES FEATURES
# ==========================================
print("\nPréparation de l'extraction de features...")

def extract_ultimate_features(df, graph, embeddings, svd_features):
    features = []
    degrees = dict(graph.degree())
    pagerank = nx.pagerank(graph, alpha=0.85, max_iter=100)

    for i, row in df.iterrows():
        u, v = int(row['source']), int(row['target'])

        common_neigh = 0
        jaccard = 0
        adamic_adar = 0
        ra_index = 0

        deg_u = degrees.get(u, 0)
        deg_v = degrees.get(v, 0)

        if graph.has_node(u) and graph.has_node(v):
            neighbors = list(nx.common_neighbors(graph, u, v))
            common_neigh = len(neighbors)
            union_size = len(set(graph[u]) | set(graph[v]))
            jaccard = common_neigh / union_size if union_size > 0 else 0

            for w in neighbors:
                d_w = degrees.get(w, 0)
                if d_w > 1:
                    adamic_adar += 1 / log(d_w)
                    ra_index += 1 / d_w

        pr_prod = pagerank.get(u, 0) * pagerank.get(v, 0)

        try:
            sp = nx.shortest_path_length(graph, u, v)
        except (nx.NetworkXNoPath, nx.NodeNotFound):
            sp = 10

        emb_u = embeddings[u]
        emb_v = embeddings[v]

        # Similarité cosinus basée sur les embeddings LightGCN
        lightgcn_cos = np.dot(emb_u, emb_v) / (np.linalg.norm(emb_u) * np.linalg.norm(emb_v) + 1e-8)

        svd_u = svd_features[u]
        svd_v = svd_features[v]
        hadamard_svd = svd_u * svd_v

        base_feats = [common_neigh, jaccard, adamic_adar, ra_index, deg_u, deg_v, deg_u*deg_v, sp, pr_prod, lightgcn_cos]
        features.append(np.concatenate([base_feats, hadamard_svd]))

    return np.array(features)


# ==========================================
# 6. STRATIFIED K-FOLD & ENSEMBLING
# ==========================================
print("\n" + "="*50)
print("DÉBUT DU K-FOLD STRATIFIÉ")
print("="*50)

n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

y_train_full = train_data['label'].values
train_pairs = train_data[['source', 'target']].values

test_preds = np.zeros(len(test_data))
oof_preds = np.zeros(len(train_data))

svd_dict = {i: node_features_reduced[i] for i in range(len(node_features_reduced))}

for fold, (train_idx, val_idx) in enumerate(skf.split(train_pairs, y_train_full)):
    print(f"\n{'='*50}")
    print(f"FOLD {fold + 1}/{n_splits}")
    print(f"{'='*50}")

    fold_train_edges = train_data.iloc[train_idx]
    fold_val_edges = train_data.iloc[val_idx]

    G_fold = nx.Graph()
    G_fold.add_nodes_from(range(len(node_features_reduced)))

    positive_train_edges = fold_train_edges[fold_train_edges['label'] == 1][['source', 'target']].values
    G_fold.add_edges_from(positive_train_edges)

    print(f"Graphe fold: {len(G_fold.nodes())} nœuds, {len(G_fold.edges())} arêtes")

    train_edges_pos = fold_train_edges[fold_train_edges['label'] == 1]
    src = torch.LongTensor(train_edges_pos['source'].values)
    dst = torch.LongTensor(train_edges_pos['target'].values)

    edge_index = torch.stack([
        torch.cat([src, dst]),
        torch.cat([dst, src])
    ], dim=0).to(device)

    labels_tensor = torch.FloatTensor(fold_train_edges['label'].values).to(device)
    train_src_all = torch.LongTensor(fold_train_edges['source'].values).to(device)
    train_dst_all = torch.LongTensor(fold_train_edges['target'].values).to(device)

    # ==========================================
    # ENTRAÎNER LIGHTGCN
    # ==========================================
    lightgcn_embeddings, embed_model, predictor = train_lightgcn_for_fold(
        node_features_tensor,
        edge_index,
        train_src_all,
        train_dst_all,
        labels_tensor,
        epochs=2000,
        lr=0.01,
        weight_decay=1e-3,
        patience=100
    )

    # ==========================================
    # EXTRAIRE FEATURES
    # ==========================================
    print("  Extraction features Train...")
    X_train_fold = extract_ultimate_features(fold_train_edges, G_fold, lightgcn_embeddings, svd_dict)
    y_train_fold = fold_train_edges['label'].values

    print("  Extraction features Val...")
    X_val_fold = extract_ultimate_features(fold_val_edges, G_fold, lightgcn_embeddings, svd_dict)
    y_val_fold = fold_val_edges['label'].values

    print(f"  Shapes: X_train {X_train_fold.shape}, X_val {X_val_fold.shape}")

    # ==========================================
    # ENTRAÎNER XGBOOST
    # ==========================================
    print("  Entraînement XGBoost...")

    clf = xgb.XGBClassifier(
        n_estimators=1000,
        max_depth=8,
        learning_rate=0.02,
        subsample=0.8,
        colsample_bytree=0.7,
        eval_metric='auc',
        early_stopping_rounds=50,
        random_state=42,
        n_jobs=-1,
        scale_pos_weight=1.0
    )

    clf.fit(
        X_train_fold, y_train_fold,
        eval_set=[(X_val_fold, y_val_fold)],
        verbose=100
    )

    oof_preds[val_idx] = clf.predict_proba(X_val_fold)[:, 1]
    print(f"  Validation AUC: {np.mean(oof_preds[val_idx]):.4f}")

    # ==========================================
    # PRÉDICTIONS TEST
    # ==========================================
    print("  Extraction features Test...")
    X_test_fold = extract_ultimate_features(test_data, G_fold, lightgcn_embeddings, svd_dict)
    test_preds += clf.predict_proba(X_test_fold)[:, 1] / n_splits

    print(f"✓ FOLD {fold + 1} TERMINÉ")

# ==========================================
# 7. SOUMISSION
# ==========================================
print("\n" + "="*50)
print("GÉNÉRATION DE LA SOUMISSION")
print("="*50)

submission = pd.DataFrame({
    'ID': range(len(test_preds)),
    'Predicted': test_preds
})

submission.to_csv('submission_lightgcn_complete.csv', index=False)
print("\n✓ Fichier 'submission_lightgcn_complete.csv' généré !")
print(f"Prédictions: min={test_preds.min():.4f}, max={test_preds.max():.4f}, mean={test_preds.mean():.4f}")

print("\n" + "="*50)
print("STATISTIQUES FINALES")
print("="*50)
print(f"OOF predictions - min: {oof_preds.min():.4f}, max: {oof_preds.max():.4f}, mean: {oof_preds.mean():.4f}")
print(f"Test predictions - min: {test_preds.min():.4f}, max: {test_preds.max():.4f}, mean: {test_preds.mean():.4f}")

Device: cpu
Mounted at /content/drive
Chargement des données...
Nœuds: 3597, Train: 10496, Test: 3498
Mappage des IDs de nœuds pour PyTorch Geometric...
Réduction des 932 features textuelles avec SVD...
Features réduites: (3597, 64)
Variance expliquée: 0.5942

INITIALISATION DU MODÈLE LIGHTGCN

Préparation de l'extraction de features...

DÉBUT DU K-FOLD STRATIFIÉ

FOLD 1/5
Graphe fold: 3597 nœuds, 4198 arêtes
Entraînement LightGCN...
  Epoch 0000 | Loss: 0.6928 | LR: 0.010000
  Epoch 0200 | Loss: 0.2854 | LR: 0.010000
  Epoch 0400 | Loss: 0.2018 | LR: 0.010000
  Epoch 0600 | Loss: 0.1769 | LR: 0.001250
  Epoch 0800 | Loss: 0.1772 | LR: 0.000005
  Early stopping à epoch 804
  Extraction features Train...
  Extraction features Val...
  Shapes: X_train (8396, 74), X_val (2100, 74)
  Entraînement XGBoost...
[0]	validation_0-auc:0.50333
[100]	validation_0-auc:0.65582
[128]	validation_0-auc:0.65827
  Validation AUC: 0.1513
  Extraction features Test...
✓ FOLD 1 TERMINÉ

FOLD 2/5
Graphe fold:

0.837

In [13]:
import pandas as pd
import numpy as np
import networkx as nx
import xgboost as xgb
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import StratifiedKFold
from math import log
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# 0. DEVICE SETUP
# ==========================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# ==========================================
# 1. CHARGEMENT ET RÉINDEXATION DES DONNÉES
# ==========================================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

path = "/content/drive/MyDrive/MLNS/"
print("Chargement des données...")
node_info = pd.read_csv(path + 'node_information.csv', header=None, index_col=0)
train_data = pd.read_csv(path + 'train.txt', sep=' ', header=None, names=['source', 'target', 'label'])
test_data = pd.read_csv(path + 'test.txt', sep=' ', header=None, names=['source', 'target'])

print(f"Nœuds: {len(node_info)}, Train: {len(train_data)}, Test: {len(test_data)}")

print("Mappage des IDs de nœuds pour PyTorch Geometric...")
mapping = {original_id: new_id for new_id, original_id in enumerate(node_info.index)}

train_data['source'] = train_data['source'].map(mapping)
train_data['target'] = train_data['target'].map(mapping)
test_data['source'] = test_data['source'].map(mapping)
test_data['target'] = test_data['target'].map(mapping)

train_data = train_data.dropna().astype(int)
test_data = test_data.dropna().astype(int)

# ==========================================
# 2. RÉDUCTION DE DIMENSION (SVD)
# ==========================================
print("Réduction des 932 features textuelles avec SVD...")
svd = TruncatedSVD(n_components=64, random_state=42)
node_features_reduced = svd.fit_transform(node_info)
node_features_tensor = torch.FloatTensor(node_features_reduced).to(device)

print(f"Features réduites: {node_features_reduced.shape}")
print(f"Variance expliquée: {svd.explained_variance_ratio_.sum():.4f}")

# ==========================================
# 3. GNN : GAT (Graph Attention Networks)
# ==========================================
print("\n" + "="*50)
print("INITIALISATION DU MODÈLE GAT")
print("="*50)

class GAT_LinkPredictionModel(nn.Module):
    """
    Graph Attention Network pour Link Prediction

    Architecture:
    - Couche 1: GATConv avec 8 heads d'attention
    - BatchNorm + ReLU + Dropout
    - Couche 2: GATConv avec 1 head

    Avantages vs GraphSAGE:
    - Multi-head attention : chaque head apprend des patterns différents
    - Plus flexible pour capturer les interactions complexes
    - Meilleur pour link prediction (empiriquement)
    """
    def __init__(self, in_feats, h_feats, out_feats, dropout=0.4, n_heads=8):
        super(GAT_LinkPredictionModel, self).__init__()

        # Couche 1: Multi-head attention
        # in_feats -> h_feats avec 8 heads (chaque head: h_feats/8)
        self.conv1 = GATConv(
            in_feats,
            h_feats // n_heads,  # Diviser par nombre de heads
            heads=n_heads,        # 8 têtes d'attention
            dropout=dropout,
            concat=True           # Concatener les heads
        )

        # Batch Norm après concat des heads
        # (h_feats // n_heads) * n_heads = h_feats
        self.bn1 = nn.BatchNorm1d(h_feats)

        # Couche 2: Attention unique
        self.conv2 = GATConv(
            h_feats,
            out_feats,
            heads=1,              # Single head pour output
            dropout=dropout,
            concat=False
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, edge_index):
        """
        Forward pass

        Args:
            x: node features [n_nodes, in_feats]
            edge_index: edge list [2, n_edges]

        Returns:
            embeddings: node embeddings [n_nodes, out_feats]
        """
        # Couche 1: Attention multi-heads
        h = self.conv1(x, edge_index)  # [n_nodes, h_feats]
        h = self.bn1(h)
        h = F.relu(h)
        h = self.dropout(h)

        # Couche 2: Attention finale
        h = self.conv2(h, edge_index)  # [n_nodes, out_feats]

        return h


class LinkPredictor_Advanced(nn.Module):
    """
    Prédicteur de liens basé sur embeddings
    """
    def __init__(self, in_feats):
        super().__init__()
        self.W1 = nn.Linear(in_feats * 2, in_feats)
        self.W2 = nn.Linear(in_feats, 1)
        self.dropout = nn.Dropout(0.2)

    def forward(self, h_u, h_v):
        combined = torch.cat([h_u, h_v], dim=1)
        x = F.relu(self.W1(combined))
        x = self.dropout(x)
        return torch.sigmoid(self.W2(x))


# ==========================================
# 4. FONCTION D'ENTRAÎNEMENT GAT
# ==========================================
def train_gat_for_fold(
    node_features_tensor,
    edge_index,
    train_src_all,
    train_dst_all,
    labels_tensor,
    epochs=2000,
    lr=0.005,
    weight_decay=1e-4,
    patience=100
):
    """
    Entraîner le GAT pour un fold donné

    Améliorations vs GraphSAGE:
    - Learning rate adapté pour GAT
    - Early stopping avec patience
    - Gradient clipping pour stabilité
    """

    embed_model = GAT_LinkPredictionModel(
        in_feats=64,
        h_feats=128,
        out_feats=64,
        dropout=0.4,
        n_heads=8
    ).to(device)

    predictor = LinkPredictor_Advanced(64).to(device)

    # Optimiseur avec weight decay pour régularisation
    optimizer = torch.optim.Adam(
        list(embed_model.parameters()) + list(predictor.parameters()),
        lr=lr,
        weight_decay=weight_decay,
        betas=(0.9, 0.999)
    )

    # Scheduler pour réduire le learning rate (VERSION CORRIGÉE - sans verbose)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=0.5,
        patience=20
    )

    embed_model.train()
    predictor.train()

    best_loss = float('inf')
    patience_counter = 0

    print("Entraînement GAT...")
    for epoch in range(epochs):
        h = embed_model(node_features_tensor, edge_index)
        pos_score = predictor(h[train_src_all], h[train_dst_all]).squeeze()
        loss = F.binary_cross_entropy(pos_score, labels_tensor)

        optimizer.zero_grad()
        loss.backward()

        # Gradient clipping pour stabilité
        torch.nn.utils.clip_grad_norm_(
            list(embed_model.parameters()) + list(predictor.parameters()),
            max_norm=1.0
        )

        optimizer.step()
        scheduler.step(loss)

        # Early stopping
        if loss.item() < best_loss:
            best_loss = loss.item()
            patience_counter = 0
        else:
            patience_counter += 1

        if epoch % 200 == 0:
            print(f"  Epoch {epoch:04d} | Loss: {loss.item():.4f} | LR: {optimizer.param_groups[0]['lr']:.6f}")

        if patience_counter >= patience:
            print(f"  Early stopping à epoch {epoch}")
            break

    embed_model.eval()
    predictor.eval()

    with torch.no_grad():
        gat_embeddings = embed_model(node_features_tensor, edge_index).cpu().numpy()

    return gat_embeddings, embed_model, predictor


# ==========================================
# 5. EXTRACTION DES FEATURES
# ==========================================
print("\nPréparation de l'extraction de features...")

def extract_ultimate_features(df, graph, embeddings, svd_features):
    """
    Extraire les 74 features pour link prediction

    Features:
    - 10 topologiques (common neigh, Jaccard, PageRank, etc.)
    - 64 Hadamard SVD (interaction des features texte)
    """
    features = []
    degrees = dict(graph.degree())
    pagerank = nx.pagerank(graph, alpha=0.85, max_iter=100)

    for i, row in df.iterrows():
        u, v = int(row['source']), int(row['target'])

        common_neigh = 0
        jaccard = 0
        adamic_adar = 0
        ra_index = 0

        deg_u = degrees.get(u, 0)
        deg_v = degrees.get(v, 0)

        if graph.has_node(u) and graph.has_node(v):
            neighbors = list(nx.common_neighbors(graph, u, v))
            common_neigh = len(neighbors)
            union_size = len(set(graph[u]) | set(graph[v]))
            jaccard = common_neigh / union_size if union_size > 0 else 0

            for w in neighbors:
                d_w = degrees.get(w, 0)
                if d_w > 1:
                    adamic_adar += 1 / log(d_w)
                    ra_index += 1 / d_w

        pr_prod = pagerank.get(u, 0) * pagerank.get(v, 0)

        try:
            sp = nx.shortest_path_length(graph, u, v)
        except (nx.NetworkXNoPath, nx.NodeNotFound):
            sp = 10

        emb_u = embeddings[u]
        emb_v = embeddings[v]
        gat_cos = np.dot(emb_u, emb_v) / (np.linalg.norm(emb_u) * np.linalg.norm(emb_v) + 1e-8)

        svd_u = svd_features[u]
        svd_v = svd_features[v]
        hadamard_svd = svd_u * svd_v

        base_feats = [common_neigh, jaccard, adamic_adar, ra_index, deg_u, deg_v, deg_u*deg_v, sp, pr_prod, gat_cos]
        features.append(np.concatenate([base_feats, hadamard_svd]))

    return np.array(features)


# ==========================================
# 6. STRATIFIED K-FOLD & ENSEMBLING
# ==========================================
print("\n" + "="*50)
print("DÉBUT DU K-FOLD STRATIFIÉ")
print("="*50)

n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

y_train_full = train_data['label'].values
train_pairs = train_data[['source', 'target']].values

test_preds = np.zeros(len(test_data))
oof_preds = np.zeros(len(train_data))

svd_dict = {i: node_features_reduced[i] for i in range(len(node_features_reduced))}

for fold, (train_idx, val_idx) in enumerate(skf.split(train_pairs, y_train_full)):
    print(f"\n{'='*50}")
    print(f"FOLD {fold + 1}/{n_splits}")
    print(f"{'='*50}")

    fold_train_edges = train_data.iloc[train_idx]
    fold_val_edges = train_data.iloc[val_idx]

    # Construire le graphe du fold
    G_fold = nx.Graph()
    G_fold.add_nodes_from(range(len(node_features_reduced)))

    positive_train_edges = fold_train_edges[fold_train_edges['label'] == 1][['source', 'target']].values
    G_fold.add_edges_from(positive_train_edges)

    print(f"Graphe fold: {len(G_fold.nodes())} nœuds, {len(G_fold.edges())} arêtes")

    # Créer edge_index pour PyTorch Geometric
    train_edges_pos = fold_train_edges[fold_train_edges['label'] == 1]
    src = torch.LongTensor(train_edges_pos['source'].values)
    dst = torch.LongTensor(train_edges_pos['target'].values)

    edge_index = torch.stack([
        torch.cat([src, dst]),
        torch.cat([dst, src])
    ], dim=0).to(device)

    # Préparer les données d'entraînement
    labels_tensor = torch.FloatTensor(fold_train_edges['label'].values).to(device)
    train_src_all = torch.LongTensor(fold_train_edges['source'].values).to(device)
    train_dst_all = torch.LongTensor(fold_train_edges['target'].values).to(device)

    # ==========================================
    # ENTRAÎNER GAT
    # ==========================================
    gat_embeddings, embed_model, predictor = train_gat_for_fold(
        node_features_tensor,
        edge_index,
        train_src_all,
        train_dst_all,
        labels_tensor,
        epochs=2000,
        lr=0.005,
        weight_decay=1e-4,
        patience=100
    )

    # ==========================================
    # EXTRAIRE FEATURES
    # ==========================================
    print("  Extraction features Train...")
    X_train_fold = extract_ultimate_features(fold_train_edges, G_fold, gat_embeddings, svd_dict)
    y_train_fold = fold_train_edges['label'].values

    print("  Extraction features Val...")
    X_val_fold = extract_ultimate_features(fold_val_edges, G_fold, gat_embeddings, svd_dict)
    y_val_fold = fold_val_edges['label'].values

    print(f"  Shapes: X_train {X_train_fold.shape}, X_val {X_val_fold.shape}")

    # ==========================================
    # ENTRAÎNER XGBOOST
    # ==========================================
    print("  Entraînement XGBoost...")

    # Paramètres optimisés pour link prediction
    clf = xgb.XGBClassifier(
        n_estimators=1000,
        max_depth=8,
        learning_rate=0.02,
        subsample=0.8,
        colsample_bytree=0.7,
        eval_metric='auc',
        early_stopping_rounds=50,
        random_state=42,
        n_jobs=-1,
        scale_pos_weight=1.0  # À ajuster si imbalance forte
    )

    clf.fit(
        X_train_fold, y_train_fold,
        eval_set=[(X_val_fold, y_val_fold)],
        verbose=100
    )

    # Prédictions validation
    oof_preds[val_idx] = clf.predict_proba(X_val_fold)[:, 1]
    print(f"  Validation AUC: {np.mean(oof_preds[val_idx]):.4f}")

    # ==========================================
    # PRÉDICTIONS TEST
    # ==========================================
    print("  Extraction features Test...")
    X_test_fold = extract_ultimate_features(test_data, G_fold, gat_embeddings, svd_dict)
    test_preds += clf.predict_proba(X_test_fold)[:, 1] / n_splits

    print(f"✓ FOLD {fold + 1} TERMINÉ")

# ==========================================
# 7. SOUMISSION
# ==========================================
print("\n" + "="*50)
print("GÉNÉRATION DE LA SOUMISSION")
print("="*50)

submission = pd.DataFrame({
    'ID': range(len(test_preds)),
    'Predicted': test_preds
})

submission.to_csv('submission_gat_complete.csv', index=False)
print("\n✓ Fichier 'submission_gat_complete.csv' généré !")
print(f"Prédictions: min={test_preds.min():.4f}, max={test_preds.max():.4f}, mean={test_preds.mean():.4f}")

# Afficher quelques statistiques
print("\n" + "="*50)
print("STATISTIQUES FINALES")
print("="*50)
print(f"OOF predictions - min: {oof_preds.min():.4f}, max: {oof_preds.max():.4f}, mean: {oof_preds.mean():.4f}")
print(f"Test predictions - min: {test_preds.min():.4f}, max: {test_preds.max():.4f}, mean: {test_preds.mean():.4f}")

Device: cpu
Mounted at /content/drive
Chargement des données...
Nœuds: 3597, Train: 10496, Test: 3498
Mappage des IDs de nœuds pour PyTorch Geometric...
Réduction des 932 features textuelles avec SVD...
Features réduites: (3597, 64)
Variance expliquée: 0.5942

INITIALISATION DU MODÈLE GAT

Préparation de l'extraction de features...

DÉBUT DU K-FOLD STRATIFIÉ

FOLD 1/5
Graphe fold: 3597 nœuds, 4198 arêtes
Entraînement GAT...
  Epoch 0000 | Loss: 0.6975 | LR: 0.005000
  Epoch 0200 | Loss: 0.3652 | LR: 0.002500
  Epoch 0400 | Loss: 0.3315 | LR: 0.000039
  Epoch 0600 | Loss: 0.3382 | LR: 0.000000
  Early stopping à epoch 663
  Extraction features Train...
  Extraction features Val...
  Shapes: X_train (8396, 74), X_val (2100, 74)
  Entraînement XGBoost...
[0]	validation_0-auc:0.50333
[100]	validation_0-auc:0.66125
[200]	validation_0-auc:0.66363
[300]	validation_0-auc:0.66357
[328]	validation_0-auc:0.66385
  Validation AUC: 0.1390
  Extraction features Test...
✓ FOLD 1 TERMINÉ

FOLD 2/5
Gra

In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
import xgboost as xgb
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import StratifiedKFold
from math import log

# ==========================================
# 1. CHARGEMENT ET RÉINDEXATION DES DONNÉES
# ==========================================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

path = "/content/drive/MyDrive/MLNS/"
print("Chargement des données...")
node_info = pd.read_csv(path + 'node_information.csv', header=None, index_col=0)
train_data = pd.read_csv(path + 'train.txt', sep=' ', header=None, names=['source', 'target', 'label'])
test_data = pd.read_csv(path + 'test.txt', sep=' ', header=None, names=['source', 'target'])

print("Mappage des IDs de nœuds pour PyTorch Geometric...")
mapping = {original_id: new_id for new_id, original_id in enumerate(node_info.index)}

train_data['source'] = train_data['source'].map(mapping)
train_data['target'] = train_data['target'].map(mapping)
test_data['source'] = test_data['source'].map(mapping)
test_data['target'] = test_data['target'].map(mapping)

train_data = train_data.dropna().astype(int)
test_data = test_data.dropna().astype(int)

# ==========================================
# 2. RÉDUCTION DE DIMENSION (SVD)
# ==========================================
print("Réduction des 932 features textuelles avec SVD...")
svd = TruncatedSVD(n_components=64, random_state=42)
node_features_reduced = svd.fit_transform(node_info)
node_features_tensor = torch.FloatTensor(node_features_reduced)

# ==========================================
# 3. GNN : EMBEDDINGS AVANCÉS
# ==========================================
print("Préparation du GNN Avancé (PyTorch Geometric)...")

train_edges = train_data[train_data['label'] == 1]
src = torch.LongTensor(train_edges['source'].values)
dst = torch.LongTensor(train_edges['target'].values)

edge_index = torch.stack([
    torch.cat([src, dst]),
    torch.cat([dst, src])
], dim=0)

class PyG_GraphSAGEModel_Advanced(nn.Module):
    def __init__(self, in_feats, h_feats, out_feats):
        super(PyG_GraphSAGEModel_Advanced, self).__init__()
        self.conv1 = SAGEConv(in_feats, h_feats)
        self.bn1 = nn.BatchNorm1d(h_feats)
        self.conv2 = SAGEConv(h_feats, out_feats)
        self.dropout = nn.Dropout(0.4)

    def forward(self, x, edge_idx):
        h = self.conv1(x, edge_idx)
        h = self.bn1(h)
        h = F.relu(h)
        h = self.dropout(h)
        h = self.conv2(h, edge_idx)
        return h

class LinkPredictor_Advanced(nn.Module):
    def __init__(self, in_feats):
        super().__init__()
        self.W1 = nn.Linear(in_feats * 2, in_feats)
        self.W2 = nn.Linear(in_feats, 1)
        self.dropout = nn.Dropout(0.2)

    def forward(self, h_u, h_v):
        combined = torch.cat([h_u, h_v], dim=1)
        x = F.relu(self.W1(combined))
        x = self.dropout(x)
        return torch.sigmoid(self.W2(x))

embed_model = PyG_GraphSAGEModel_Advanced(in_feats=64, h_feats=128, out_feats=64)
predictor = LinkPredictor_Advanced(64)

optimizer = torch.optim.Adam(
    list(embed_model.parameters()) + list(predictor.parameters()),
    lr=0.005,
    weight_decay=1e-4
)

print("Entraînement de GraphSAGE (2000 epochs)...")
labels_tensor = torch.FloatTensor(train_data['label'].values)
train_src_all = torch.LongTensor(train_data['source'].values)
train_dst_all = torch.LongTensor(train_data['target'].values)

embed_model.train()
predictor.train()

for epoch in range(2501):
    h = embed_model(node_features_tensor, edge_index)
    pos_score = predictor(h[train_src_all], h[train_dst_all]).squeeze()
    loss = F.binary_cross_entropy(pos_score, labels_tensor)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 100 == 0:
        print(f"  Epoch {epoch:04d} | Loss: {loss.item():.4f}")

embed_model.eval()
predictor.eval()

with torch.no_grad():
    gnn_embeddings = embed_model(node_features_tensor, edge_index).numpy()

# ==========================================
# 4. EXTRACTION DES FEATURES (Version 0.835)
# ==========================================
print("Préparation de l'extraction de features...")

def extract_ultimate_features(df, graph, embeddings, svd_features):
    features = []
    degrees = dict(graph.degree())
    pagerank = nx.pagerank(graph, alpha=0.85)

    for i, row in df.iterrows():
        u, v = int(row['source']), int(row['target'])

        common_neigh = 0
        jaccard = 0
        adamic_adar = 0
        ra_index = 0

        deg_u = degrees.get(u, 0)
        deg_v = degrees.get(v, 0)

        if graph.has_node(u) and graph.has_node(v):
            neighbors = list(nx.common_neighbors(graph, u, v))
            common_neigh = len(neighbors)
            union_size = len(set(graph[u]) | set(graph[v]))
            jaccard = common_neigh / union_size if union_size > 0 else 0

            for w in neighbors:
                d_w = degrees.get(w, 0)
                if d_w > 1:
                    adamic_adar += 1 / log(d_w)
                    ra_index += 1 / d_w

        pr_prod = pagerank.get(u, 0) * pagerank.get(v, 0)

        # Le fameux Shortest Path avec le fix NodeNotFound (mais SANS remove_edge)
        try:
            sp = nx.shortest_path_length(graph, u, v)
        except (nx.NetworkXNoPath, nx.NodeNotFound):
            sp = 10

        emb_u = embeddings[u]
        emb_v = embeddings[v]
        gnn_cos = np.dot(emb_u, emb_v) / (np.linalg.norm(emb_u) * np.linalg.norm(emb_v) + 1e-8)

        svd_u = svd_features[u]
        svd_v = svd_features[v]
        hadamard_svd = svd_u * svd_v

        base_feats = [common_neigh, jaccard, adamic_adar, ra_index, deg_u, deg_v, deg_u*deg_v, sp, pr_prod, gnn_cos]
        features.append(np.concatenate([base_feats, hadamard_svd]))

    return np.array(features)

# ==========================================
# 5. ENTRAÎNEMENT STRATIFIED K-FOLD & ENSEMBLING
# ==========================================
print("Début du K-Fold Out-Of-Fold (OOF)...")

n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

y_train_full = train_data['label'].values
train_pairs = train_data[['source', 'target']].values

test_preds = np.zeros(len(test_data))
oof_preds = np.zeros(len(train_data))

svd_dict = {i: node_features_reduced[i] for i in range(len(node_features_reduced))}

for fold, (train_idx, val_idx) in enumerate(skf.split(train_pairs, y_train_full)):
    print(f"\n--- FOLD {fold + 1}/{n_splits} ---")

    fold_train_edges = train_data.iloc[train_idx]
    fold_val_edges = train_data.iloc[val_idx]

    G_fold = nx.Graph()
    # Le fix essentiel pour éviter le plantage
    G_fold.add_nodes_from(range(len(node_features_reduced)))

    positive_train_edges = fold_train_edges[fold_train_edges['label'] == 1][['source', 'target']].values
    G_fold.add_edges_from(positive_train_edges)

    print("  Extraction features Train...")
    X_train_fold = extract_ultimate_features(fold_train_edges, G_fold, gnn_embeddings, svd_dict)
    y_train_fold = fold_train_edges['label'].values

    print("  Extraction features Val...")
    X_val_fold = extract_ultimate_features(fold_val_edges, G_fold, gnn_embeddings, svd_dict)
    y_val_fold = fold_val_edges['label'].values

    # Paramètres XGBoost de ton record
    clf = xgb.XGBClassifier(
        n_estimators=1000,
        max_depth=8,
        learning_rate=0.02,
        subsample=0.8,
        colsample_bytree=0.7,
        eval_metric='auc',
        early_stopping_rounds=50
    )

    clf.fit(
        X_train_fold, y_train_fold,
        eval_set=[(X_val_fold, y_val_fold)],
        verbose=100
    )

    oof_preds[val_idx] = clf.predict_proba(X_val_fold)[:, 1]

    print("  Extraction features Test...")
    X_test_fold = extract_ultimate_features(test_data, G_fold, gnn_embeddings, svd_dict)
    test_preds += clf.predict_proba(X_test_fold)[:, 1] / n_splits

# ==========================================
# 6. SOUMISSION
# ==========================================
submission = pd.DataFrame({
    'ID': range(len(test_preds)),
    'Predicted': test_preds
})
submission.to_csv('submission_record.csv', index=False)
print("\nFichier 'submission_record.csv' généré ! Tu retrouves ton meilleur code.")

Mounted at /content/drive
Chargement des données...
Mappage des IDs de nœuds pour PyTorch Geometric...
Réduction des 932 features textuelles avec SVD...
Préparation du GNN Avancé (PyTorch Geometric)...
Entraînement de GraphSAGE (2000 epochs)...
  Epoch 0000 | Loss: 0.7038
  Epoch 0100 | Loss: 0.1169
  Epoch 0200 | Loss: 0.0586
  Epoch 0300 | Loss: 0.0389
  Epoch 0400 | Loss: 0.0288
  Epoch 0500 | Loss: 0.0274
  Epoch 0600 | Loss: 0.0270
  Epoch 0700 | Loss: 0.0212
  Epoch 0800 | Loss: 0.0206
  Epoch 0900 | Loss: 0.0211
  Epoch 1000 | Loss: 0.0175
  Epoch 1100 | Loss: 0.0145
  Epoch 1200 | Loss: 0.0166
  Epoch 1300 | Loss: 0.0155
  Epoch 1400 | Loss: 0.0199
  Epoch 1500 | Loss: 0.0170
  Epoch 1600 | Loss: 0.0161
  Epoch 1700 | Loss: 0.0150
  Epoch 1800 | Loss: 0.0158
  Epoch 1900 | Loss: 0.0133
  Epoch 2000 | Loss: 0.0135
  Epoch 2100 | Loss: 0.0163
  Epoch 2200 | Loss: 0.0122
  Epoch 2300 | Loss: 0.0115
  Epoch 2400 | Loss: 0.0126
  Epoch 2500 | Loss: 0.0139
Préparation de l'extraction